# Ensemble Methods
We are going to be testing ensemble methods for predicting stock prices

## ESNForest MSFT Experiment

Train an ESNForest ensemble on MSFT time-series features generated by `make_single_stock_df`.

In [5]:
# Imports and notebook setup
from pathlib import Path
import os
import sys

import torch
from torch.utils.data import TensorDataset

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from my_engine.config import ModelConfig, TrainerConfig, MetricsConfig
from my_engine.data import get_dataloaders as get_data_loaders
from my_engine.financial_data import make_single_stock_df
from my_engine.trainer import Trainer, RidgeRegressionTrainer
from my_engine.utils import build_model, make_optimizer, get_direction_accuracy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
DEVICE

device(type='cuda')

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Load MSFT data and convert targets to column vectors for regression
WINDOW_SIZE = 30

train_ds, val_ds, test_ds, scaler, features = make_single_stock_df(
    ticker="MSFT",
    period="5y",
    train_split=0.8,
    val_split=0.1,
    window_size=WINDOW_SIZE,
)


def with_column_targets(dataset):
    X, y = dataset.tensors
    if y.ndim == 1:
        y = y.unsqueeze(-1)
    return TensorDataset(X, y)


train_ds = with_column_targets(train_ds)
val_ds = with_column_targets(val_ds)
test_ds = with_column_targets(test_ds) if test_ds is not None else None

num_inputs = len(features)
print(f"features ({num_inputs}): {features}")
print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds) if test_ds is not None else 0}")

[*********************100%***********************]  1 of 1 completed


features (11): ['log_returns', 'log_volume', 'log_intraday_chng', '10_log_returns_ma', '20_log_returns_ma', '50_log_returns_ma', 'open_close', 'overnight_gap', 'ret_1', 'ret_5', 'variance']
train=933, val=211, test=91


In [7]:
# Dataloaders for time-series training
trainer_config = TrainerConfig(
    device=DEVICE,
)

train_loader, val_loader, test_loader = get_data_loaders(
    train_ds=train_ds,
    eval_ds=val_ds,
    test_ds=test_ds,
    train_batch_size=trainer_config.trainer_batch_size,
    eval_batch_size=trainer_config.evaluator_batch_size,
    test_batch_size=trainer_config.evaluator_batch_size,
    time_series=True,
)

In [44]:
# Build ESNForest from ModelConfig and build_model
esn_forest_config = ModelConfig(
    model_type="esn_forest",
    hidden_units=[128],
    dropout=[0.2],
    resevior_sizes=[25, 50 , 75, 100, 200, 250],
    esn_depths=[1, 2, 3, 4, 5, 6, 7],
    leak_rate_range=(0.2, 1.0),
    reservoir_sparsity_range=(0.5, 0.99),
    spectral_radius_range=(0.5, 1.2),
    input_scale_range=(0.1, 0.8),
    number_esns=20,
)

esn_forest_model = build_model(
    input_spec=num_inputs,
    num_outputs=1,
    config=esn_forest_config,
)

print(esn_forest_model)
print("parameters:", esn_forest_model.num_parameters())
for idx, member_config in enumerate(esn_forest_model.member_configs):
    print(
        f"member {idx}: type={member_config.model_type}, "
        f"reservoir={member_config.reservoir_size}, "
        f"depth={member_config.rnn_num_layers}, "
        f"leak={member_config.leak_rate:.3f}, "
        f"sparsity={member_config.reservoir_sparsity:.3f}, "
        f"radius={member_config.spectral_radius:.3f}, "
        f"input_scale={member_config.input_scale:.3f}"
    )

ESNForest(input=11, members=20, out=1)
parameters: (2402245, 0)
member 0: type=deep_esn, reservoir=75, depth=2, leak=0.947, sparsity=0.588, radius=1.020, input_scale=0.656
member 1: type=deep_esn, reservoir=25, depth=5, leak=0.717, sparsity=0.911, radius=0.852, input_scale=0.334
member 2: type=deep_esn, reservoir=25, depth=5, leak=0.554, sparsity=0.662, radius=1.118, input_scale=0.453
member 3: type=deep_esn, reservoir=25, depth=4, leak=0.881, sparsity=0.530, radius=1.074, input_scale=0.536
member 4: type=deep_esn, reservoir=200, depth=2, leak=0.837, sparsity=0.915, radius=0.821, input_scale=0.210
member 5: type=esn, reservoir=100, depth=1, leak=0.803, sparsity=0.956, radius=1.004, input_scale=0.296
member 6: type=esn, reservoir=250, depth=1, leak=0.846, sparsity=0.702, radius=0.987, input_scale=0.658
member 7: type=deep_esn, reservoir=250, depth=6, leak=0.804, sparsity=0.705, radius=0.927, input_scale=0.261
member 8: type=deep_esn, reservoir=50, depth=4, leak=0.617, sparsity=0.727, ra

In [45]:
# Train ESNForest with TrainerConfig and Trainer
criterion = torch.nn.MSELoss()

trainer = RidgeRegressionTrainer(
    model=esn_forest_model,
    criterion=criterion,
    config=trainer_config,
    ridge_alpha=10
)
esn_forest_results = trainer.fit(train_loader, val_loader)

esn_forest_results

{'train_loss': 0.8299911805415587,
 'train_acc': 0.0,
 'val_loss': 0.8738547563552856,
 'val_acc': 0.0}

In [11]:
# Evaluate on the held-out MSFT test split
if test_loader is not None:
    test_loss, _, _ = trainer.validate(test_loader)
    print(f"ESNForest MSFT test MSE: {test_loss:.6f}")
else:
    print("No test split was created.")

ValueError: not enough values to unpack (expected 3, got 2)

In [48]:
dir_acc, true_dirs, pred_dirs = get_direction_accuracy(esn_forest_model, train_loader, scaler, log_returns_idx=0)
print(f"Directional Accuracy: {dir_acc:.2%}")

Directional Accuracy: 71.70%
